# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Data-source access blocker: research-collection.ethz.ch bitstream content endpoint returned HTTP 429 (no Retry-After) on every request for 85+ minutes (14:09Z-15:26Z), verified from two different Colab VMs/egress IPs, including 7 bounded backoff attempts over 23 minutes. This is the only public host for the dataset; no alternate route exists. Not a missing-material conclusion: record metadata (uuids/sizes/MD5s) remains pinned and valid. R environment verified feasible (R 4.6.1, all 9 packages installed OK). Notebook patched with evidence-backed eth_get backoff and kept as UNVERIFIED deliverable; host queue may publish it as an unverified reference. Resume with a single Range probe; run the full notebook once the endpoint recovers.

Attempt status: blocked. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Extracting flowering phenology from grassland species mixtures using time-lapse cameras

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-e9687bee7cc9400e62453c3aeb528a7d)


# Flowering phenology from time-lapse cameras — single-plot reproduction

Reproduction of the core flower-cover time series (FCTS) pipeline of:

> Andreatta D., Bachofen C., Dalponte M., Klaus V.H., Buchmann N.C. (2023).
> **Extracting flowering phenology from grassland species mixtures using time-lapse cameras.**
> *Remote Sensing of Environment* 298:113835. DOI: [10.1016/j.rse.2023.113835](https://doi.org/10.1016/j.rse.2023.113835)

Author tutorial code (R): [`andreattad/Flower_covers_phenocams`](https://github.com/andreattad/Flower_covers_phenocams) @ commit `7cf58731236438ad1200aac2553c9e48461ee9bf` (R 4.3.0 tutorials `Phase_1.md`–`Phase_4.md`, `image_classification.md`).
Data: ETH Research Collection, DOI [10.3929/ethz-b-000634004](https://doi.org/10.3929/ethz-b-000634004) (CC BY-ND 4.0, ~19.4 GB total; we fetch only a minimal subset).

**Scope — PARTIAL DEMONSTRATION (one plot, one season):** train the paper's random-forest pixel classifier from the shared labelled pixels (SFFS-selected 11 bands, downscaling DF=4, GLCM window WS=11), classify the brightness/contrast-filtered 2014 season images (DOY 113–150) of ONE plot inside its region of interest (ROI), compute per-image class percentages, then smooth the FCTS (`tsclean` + daily means + `loess`) and extract onset/peak/end flowering dates for the sown species. The paper's 89 plots, the DF–WS sweep, and the SFFS re-selection are out of scope; we use the authors' recorded result (DF=4, WS=11; 11 bands listed in their Phase 3/4 code).

**Execution status:** CPU runtime (no GPU is used — the method is CPU image processing + randomForest). The validation status is stated in the validation record at the end of this notebook and must come from a fresh top-to-bottom Colab run; see also the run report.

**AI-assisted orchestration notice / AI支援に関する注意:** The scientific operations run the **authors' own R code** (pinned commit, cited per cell) through `Rscript`. The Colab orchestration (downloads, range-request extraction, glue) and small documented adaptations (listed in each section; e.g. brightness/contrast computed on ROI pixels without `phenopix::extractVIs`, manual F1 instead of `crfsuite`) are AI-generated and were validated on the example below, but untested behaviour is not guaranteed. 本ノートブックは著者のRコードを`Rscript`で実行し、実行環境の構築と一部の代替実装（明記）はAIが追加したものです。

**Runtime:** select **CPU** (Runtime → Change runtime type → CPU). No GPU is used.
**Expected duration:** ~45–90 min end-to-end (package install ~5–15 min; ~0.3–1 GB of selective downloads; feature extraction and classification of ~100–300 images). Free-tier availability may vary.
**Data boundary:** all dataset bytes are downloaded and processed inside this Colab VM under `/content/fcpheno`; nothing is required from the machine running this notebook.

## 1. Environment check

**Purpose / expected result:** verify that this Colab VM provides `Rscript` (the standard Colab CPU image ships R) and enough disk for the selective downloads. The cell fails loudly with instructions if R is missing.

*このVMにRがあるかとディスク空き容量を確認します。*

In [ ]:
import os, sys, subprocess, shutil

assert os.path.exists('/usr/bin/Rscript'), (
    '/usr/bin/Rscript not found: select a standard Colab runtime image that includes R, '
    'or install R via apt in a setup cell.')
print(subprocess.run(['/usr/bin/Rscript', '--version'], capture_output=True, text=True).stdout.strip())
print('CPUs:', os.cpu_count())
st = os.statvfs('/content')
print('Free disk: %.1f GB' % (st.f_bavail * st.f_frsize / 1e9))
import requests
print('python', sys.version.split()[0], '| requests', requests.__version__)

## 2. R packages

**Purpose / expected result:** install the R packages used by the author pipeline from the Posit Package Manager Ubuntu 22.04 (jammy) **binary** repository (fast, matches the VM's R). Versions are printed for the validation record; we do not pin versions because the VM's R version is chosen by Colab, and we record whatever was installed.

Packages used by the authors' path: `raster`, `terra`, `glcm`, `randomForest`, `forecast`, `zoo`, `dplyr`, `sp`, `stringr`, plus `scales` (`rescale()` used by the authors' `findpm`). Skipped author packages and their documented replacements:

| Author package | Role here | Replacement |
|---|---|---|
| `phenopix` | `extractVIs` brightness/contrast | reimplemented on ROI pixels (same formula, see §7) |
| `rgdal` | retired/spatial IO | not needed (`sp`/`terra` cover it) |
| `crfsuite` | `crf_evaluation` F1 | manual F1 from the confusion table |
| `varSel` | SFFS selection | the authors' recorded SFFS result (11 bands, cited in their Phase 3/4 code) |

*著者パッケージのうち置き換えたものは上表の通りです(いずれも明記)。*

In [ ]:
script = r'''
k <- c("raster","terra","glcm","randomForest","forecast","zoo","dplyr","sp","stringr","scales")
ok <- sapply(k, requireNamespace, quietly=TRUE)
miss <- k[!ok]
if (length(miss)) {
  options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
  install.packages(miss, Ncpus = 2)
}
for (p in k) cat(p, as.character(packageVersion(p)), "\n")
cat(R.version.string, "\n")
'''
p = subprocess.run(['/usr/bin/Rscript', '-e', script], capture_output=True, text=True)
print(p.stdout)
if p.returncode != 0:
    print(p.stderr[-3000:])
    raise RuntimeError('R package installation failed; see stderr above.')

## 3. Labels / ROIs / plot composition download

**Provenance:** ETH Research Collection record `10.3929/ethz-b-000634004` (item uuid `e1341805-51f7-4577-95cd-1c9a807fa28b`), file `ROIS_LABELS_COMPOSITION.zip`, bitstream uuid `be0cf5ef-e646-4f8c-83bc-ca2279c0deae`, size 75,057,306 bytes, MD5 `181099ca823025019e84977636d51e9f` (pinned from the record's public file-list API). CC BY-ND 4.0.

**Expected result:** the MD5-verified archive is extracted on the VM; its member list is printed (contains the labelled pixel table, per-plot ROI `Rdata` files, and the plot composition table).

During validation attempts on 2026-10-09 the ETH Research Collection content endpoint returned HTTP 429 (no `Retry-After`) continuously for >80 minutes from two different Colab VMs, so the download cells use bounded exponential backoff and fail with an explicit error if the limit persists.

*ラベル・ROI・構成表を含む75MBのzipをMD5検証付きでダウンロード・展開します。検証時に当エンドポイントが長時間429を返したため、指数バックオフ付きで再試行します。*

In [ ]:
import requests, hashlib, zipfile, os, time

def eth_get(url, headers=None, stream=False, timeout=300, attempts=8,
            base_wait=30, max_wait=300, tag='[eth]'):
    """GET with bounded exponential backoff for 429/5xx; respects Retry-After when present."""
    last = None
    for a in range(attempts):
        r = requests.get(url, headers=headers, stream=stream, timeout=timeout)
        if r.status_code == 200:
            return r
        last = r
        ra = r.headers.get('Retry-After')
        wait = int(ra) if (ra or '').isdigit() else min(max_wait, base_wait * (2 ** a))
        print(f'{tag} HTTP {r.status_code}, attempt {a+1}/{attempts}, waiting {wait}s', flush=True)
        if r.status_code not in (429, 500, 502, 503, 504):
            r.raise_for_status()
        time.sleep(wait)
    last.raise_for_status()

BASE = 'https://www.research-collection.ethz.ch/server/api/core/bitstreams/'
ROIS_UUID = 'be0cf5ef-e646-4f8c-83bc-ca2279c0deae'
ROIS_MD5 = '181099ca823025019e84977636d51e9f'
url = BASE + ROIS_UUID + '/content'

os.makedirs('/content/fcpheno/data', exist_ok=True)
zp = '/content/fcpheno/data/ROIS_LABELS_COMPOSITION.zip'
h = hashlib.md5()
resp = eth_get(url, stream=True, timeout=600, tag='[labels]')
with resp:
    with open(zp, 'wb') as f:
        for chunk in resp.iter_content(1 << 20):
            f.write(chunk); h.update(chunk)
size = os.path.getsize(zp)
print('downloaded', size, 'bytes; MD5 ok:', h.hexdigest() == ROIS_MD5)
assert size == 75057306 and h.hexdigest() == ROIS_MD5, 'labels zip failed checksum'

with zipfile.ZipFile(zp) as z:
    names = z.namelist()
    print('members:', len(names))
    for n in names[:80]:
        print('  ', n, z.getinfo(n).file_size)
    z.extractall('/content/fcpheno/data/rois_labels')
print('extracted to /content/fcpheno/data/rois_labels')

## 4. Labelled pixels and plot selection

**Purpose / expected result:** inspect `Phase_1_labelled.csv` (the authors' shared labelled pixel table: pixel `x,y`, class `type`, source image `im`) and the plot composition table, then choose the demonstration plot **deterministically from input metadata only** (never from results): the first plot ID, in the authors' `plotIDs` order (Phase_1.md), whose sown species list includes *Leucanthemum vulgare* (the most accurately predicted flower species in the paper's validation); if none, fall back to *Knautia arvensis*, then *Ranunculus acris*. The chosen plot's sown list is printed.

*ラベル表のスキーマ確認と、構成表に基づく決定論的なデモ用プロット選択を行います。*

In [ ]:
import glob, json, pandas as pd

labs_path = glob.glob('/content/fcpheno/data/rois_labels/**/Phase_1_labelled.csv', recursive=True)
assert labs_path, 'Phase_1_labelled.csv not found in the labels zip'
labs = pd.read_csv(labs_path[0])
print('labelled pixels:', labs.shape)
print(labs.head(5))
print('classes:', sorted(labs['type'].astype(str).unique()))
labs['im'] = labs['im'].astype(str).map(lambda s: s.split('/')[-1])
print('labelled images:', labs['im'].nunique())

comp_paths = [p for p in glob.glob('/content/fcpheno/data/rois_labels/**/*.csv', recursive=True)
              if 'composition' in p.lower()]
assert comp_paths, 'plot composition csv not found'
comp = pd.read_csv(comp_paths[0])
print('composition:', comp.shape, list(comp.columns))
print(comp.head(3))

spcol = 'sp_list' if 'sp_list' in comp.columns else None
assert spcol, f"no 'sp_list' column; columns are {list(comp.columns)}"
plotcol = 'plot' if 'plot' in comp.columns else [c for c in comp.columns if 'plot' in c.lower()][0]
comp['_pid'] = comp[plotcol].astype(str).str.zfill(3)

# author's plotIDs order (Phase_1.md)
PLOT_IDS = ["001","002","003","005","007","008","009","010","011","013","016","017","018","021",
 "024","025","026","028","030","035","037","039","040","042","043","044","045","046","048","049",
 "051","053","054","056","057","058","059","060","062","064","065","067","070","071","073","074",
 "075","077","079","080","081","082","083","084","085","088","090","091","092","093","094","095",
 "097","099","100","102","103","105","108","109","110","111","113","114","115","116","119","120",
 "121","125","128","129","130","131","133","135","136","137","138"]
comp_by_id = comp.set_index('_pid')
chosen, reason = None, None
for sp, why in [('Leucanthemum', 'sown Leucanthemum vulgare'),
                ('Knautia', 'sown Knautia arvensis'),
                ('Ranunculus', 'sown Ranunculus acris')]:
    for pid in PLOT_IDS:
        if pid in comp_by_id.index and sp in str(comp_by_id.loc[pid, spcol]):
            chosen, reason = pid, why
            break
    if chosen: break
if not chosen:
    chosen = next(pid for pid in PLOT_IDS if pid in comp_by_id.index)
    reason = 'first author plot id (no sown flower match)'
print('CHOSEN PLOT:', chosen, '-', reason)
print('sown list:', comp_by_id.loc[chosen, spcol])

json.dump({'plot': chosen, 'reason': reason}, open('/content/fcpheno/plot_choice.json', 'w'))
PLOT = chosen

## 5. Selective image retrieval over HTTP Range

**Purpose / expected result:** the three image archives (`IMGS_PLOTS_001_049.zip` 7.16 GB, `IMGS_PLOTS_051_093.zip` 6.91 GB, `IMGS_PLOTS_094_138.zip` 6.72 GB; MD5s pinned from the record API) are far too large to download for a single-plot demo. Instead we read each ZIP **central directory** with a few HTTP `Range` requests, select only the needed members, and fetch just those byte ranges into a sparse local ZIP (entries are then extracted with `zipfile`, which also verifies each entry's CRC). Expected transfer: a few hundred MB instead of ~20 GB.

If the server rejects range requests, the cell falls back to a **disclosed heavy fallback**: full download of the needed archive(s) one at a time (deleted after extraction).

Bitstream uuids (record metadata): `001_049` → `12b9c398-53c1-4ea3-95aa-b2b144e9f63a`; `051_093` → `7b350c0a-66c9-4a63-98a9-69633915ac4f`; `094_138` → `0ed8ea1c-0a25-4d9f-8b13-0a2f82dadae1`.

*7GBのzipをRangeリクエストで必要部分のみ取得します(サーバが非対応の場合は全体ダウンロードにフォールバックし、その旨を明示します)。*

In [ ]:
import requests, struct, re, os, json, datetime, shutil, zipfile
from concurrent.futures import ThreadPoolExecutor

SESSION = requests.Session()
ADAPTER = requests.adapters.HTTPAdapter(pool_connections=8, pool_maxsize=8, max_retries=3)
SESSION.mount('https://', ADAPTER)
BASE = 'https://www.research-collection.ethz.ch/server/api/core/bitstreams/'
ZIP_UUIDS = {
    'A': '12b9c398-53c1-4ea3-95aa-b2b144e9f63a',  # plots 001-049
    'B': '7b350c0a-66c9-4a63-98a9-69633915ac4f',  # plots 051-093
    'C': '0ed8ea1c-0a25-4d9f-8b13-0a2f82dadae1',  # plots 094-138
}
os.makedirs('/content/fcpheno/data', exist_ok=True)

def fetch(url, start, end):
    r = eth_get(url, headers={'Range': f'bytes={start}-{end}'}, timeout=300,
                tag=f'[range {start//1000000}MB]')
    return r.content

def head_size(url):
    r = SESSION.get(url, headers={'Range': 'bytes=0-0'}, timeout=120)
    cr = r.headers.get('Content-Range', '')
    return (int(cr.rsplit('/', 1)[-1]), r.status_code == 206) if cr else (None, False)

class RemoteZip:
    def __init__(self, url):
        self.url = url
        self.total, self.ranges_ok = head_size(url)
        if not self.total or not self.ranges_ok:
            raise RuntimeError('range requests not supported by the server')
        tail = fetch(url, max(0, self.total - 65536), self.total - 1)
        i = tail.rfind(b'PK\x05\x06')
        assert i >= 0, 'no EOCD'
        (_, _, _, _, _, n_total, cd_size, cd_ofs,
         _) = struct.unpack('<IHHHHIIH', tail[i:i+22])
        if cd_ofs == 0xFFFFFFFF or n_total == 0xFFFF:
            j = tail.rfind(b'PK\x06\x06')
            assert j >= 0, 'no ZIP64 EOCD'
            n_total = struct.unpack('<Q', tail[j+32:j+40])[0]
            cd_size, cd_ofs = struct.unpack('<QQ', tail[j+40:j+56])
        self.cd_size, self.cd_ofs, self.n_total = cd_size, cd_ofs, n_total
        cd = fetch(url, cd_ofs, cd_ofs + cd_size - 1)
        self.entries = {}
        k = 0
        while k < len(cd) - 4 and len(self.entries) < n_total:
            assert cd[k:k+4] == b'PK\x01\x02', f'bad CD sig at {k}'
            (_, _, flags, method, _, _, crc, csize, usize,
             nlen, elen, clen, _, _, _, lho) = struct.unpack('<HHHHHHIIIHHHHHII', cd[k+4:k+46])
            name = cd[k+46:k+46+nlen].decode('utf-8', 'replace')
            self.entries[name] = (method, crc, csize, usize, lho)
            k += 46 + nlen + elen + clen
        self.tail = fetch(url, cd_ofs, self.total - 1)  # central dir + zip64/EOCD records

    def build_sparse(self, wanted_names, sparse_path):
        with open(sparse_path, 'wb') as f:
            f.truncate(self.total)
            for name in wanted_names:
                method, crc, csize, usize, lho = self.entries[name]
                hdr = fetch(self.url, lho, lho + 29)
                nlen, elen = struct.unpack('<HH', hdr[26:30])
                nameextra = fetch(self.url, lho + 30, lho + 29 + nlen + elen)
                ds = lho + 30 + nlen + elen
                data = fetch(self.url, ds, ds + csize - 1)
                f.seek(lho); f.write(hdr + nameextra + data)
            f.seek(self.cd_ofs); f.write(self.tail)
        return len(wanted_names)

def full_download(uuid_tag, dest_path):
    url = BASE + ZIP_UUIDS[uuid_tag] + '/content'
    print(f'FALLBACK: downloading full archive {uuid_tag} (heavy, ~7 GB)...')
    r = eth_get(url, stream=True, timeout=7200, attempts=6, tag=f'[full {uuid_tag}]')
    with r:
        done = 0
        with open(dest_path, 'wb') as f:
            for chunk in r.iter_content(1 << 22):
                f.write(chunk); done += len(chunk)
                if done % (1 << 28) < (1 << 22):
                    print(f'  {done/1e9:.2f} GB', flush=True)
    print('fallback download complete:', dest_path)

ZIPS = {}
for tag, u in ZIP_UUIDS.items():
    try:
        ZIPS[tag] = RemoteZip(BASE + u + '/content')
        jpgs = [e for e in ZIPS[tag].entries if e.lower().endswith('.jpg')]
        print(f'zip {tag}: {len(ZIPS[tag].entries)} entries, {len(jpgs)} jpg, '
              f'size {ZIPS[tag].total/1e9:.2f} GB, ranges=True')
    except RuntimeError as exc:
        ZIPS[tag] = None
        print(f'zip {tag}: range requests unavailable ({exc}); full-download fallback will be used')
json.dump({t: (ZIPS[t].total if ZIPS[t] else None) for t in ZIPS},
          open('/content/fcpheno/state_zips.json', 'w'))

def ensure_members(tag, wanted_names, dest_dir, manifest):
    """Extract the wanted archive members to dest_dir (flat), via ranges or full download."""
    wanted = set(wanted_names)
    os.makedirs(dest_dir, exist_ok=True)
    if ZIPS[tag] is not None:
        z = ZIPS[tag]
        members = [n for n in z.entries if n.lower().endswith('.jpg') and n.split('/')[-1] in wanted]
        if not members:
            return []
        sparse = f'/content/fcpheno/data/sparse_{tag}.zip'
        z.build_sparse(members, sparse)
        with zipfile.ZipFile(sparse) as zf:
            for m in members:
                zf.extract(m, '/content/fcpheno/data/_x')  # CRC verified here
                info = z.entries[m]
                manifest.append({'zip': tag, 'name': m, 'csize': info[2], 'usize': info[3]})
        os.remove(sparse)
    else:
        fp = f'/content/fcpheno/data/full_{tag}.zip'
        if not os.path.exists(fp):
            full_download(tag, fp)
        with zipfile.ZipFile(fp) as zf:
            for m in zf.namelist():
                if m.lower().endswith('.jpg') and m.split('/')[-1] in wanted:
                    zf.extract(m, '/content/fcpheno/data/_x')
                    manifest.append({'zip': tag, 'name': m,
                                     'csize': zf.getinfo(m).compress_size,
                                     'usize': zf.getinfo(m).file_size})
    got = []
    for root, _, fs in os.walk('/content/fcpheno/data/_x'):
        for f in fs:
            if f.lower().endswith('.jpg'):
                shutil.move(os.path.join(root, f), os.path.join(dest_dir, f))
                got.append(f)
    shutil.rmtree('/content/fcpheno/data/_x', ignore_errors=True)
    return got

## 6. Download the labelled images (training set)

**Purpose / expected result:** the ~300 images that contain the authors' labelled pixels (image names from `Phase_1_labelled.csv`) may span the three plot-block archives. We range-fetch exactly those members (plus nothing else) into sparse ZIPs and extract them with CRC verification; a JPEG magic-byte check follows. A member manifest (name, compressed/uncompressed size) is kept for provenance.

*学習用のラベル付き画像(~300枚)のみをRange取得・CRC検証して展開します。*

In [ ]:
import pandas as pd

lab_names = set(labs['im'].unique())
manifest = []
got = []
jobs = {}
for tag, z in ZIPS.items():
    if z is None:
        jobs[tag] = None
        continue
    members = [n for n in z.entries if n.lower().endswith('.jpg') and n.split('/')[-1] in lab_names]
    if members:
        jobs[tag] = members
print('labelled images per block zip:', {t: (len(m) if m else 'fallback') for t, m in jobs.items()})
with ThreadPoolExecutor(max_workers=3) as ex:
    for res in ex.map(lambda kv: ensure_members(kv[0], lab_names,
                                               '/content/fcpheno/data/train_imgs', manifest),
                      jobs.items()):
        got += res
imgs_on_disk = set(os.listdir('/content/fcpheno/data/train_imgs'))
print('labelled images extracted:', len(imgs_on_disk), 'of', len(lab_names))
assert imgs_on_disk == lab_names, (
    f'missing labelled images: {sorted(lab_names - imgs_on_disk)[:5]}')
bad = [f for f in imgs_on_disk
       if open(f'/content/fcpheno/data/train_imgs/{f}', 'rb').read(3) != b'\xff\xd8\xff']
assert not bad, f'non-JPEG payloads: {bad[:5]}'
pd.DataFrame(manifest).to_csv('/content/fcpheno/data/labelled_manifest.csv', index=False)
print('JPEG magic ok for all', len(imgs_on_disk), 'labelled images')

## 7. Download the chosen plot's season images

**Purpose / expected result:** list the chosen plot's camera images inside its block archive, keep those from the paper's season window (2014, day-of-year 113–150, per Phase_1.md `start=113; end=150`), and range-fetch exactly those members. If the archive stores more images in the window than the paper's per-plot selection implies (the authors report ~137 images/plot), the cell stops with a clear count instead of silently downloading gigabytes.

*選択プロットの季節窓(DOY 113–150)の画像のみを取得します。*

In [ ]:
import re, datetime

tag = 'A' if int(PLOT) <= 49 else ('B' if int(PLOT) <= 93 else 'C')
z = ZIPS[tag]
if z is None:
    season = None
    print('range path unavailable for zip', tag, '- listing requires the fallback download')
else:
    pat = re.compile(r'(?:^|/)(\d{3})/[^/]*\.jpg$', re.I)
    plot_entries = [n for n in z.entries if pat.search(n) and pat.search(n).group(1) == PLOT]
    print('plot', PLOT, 'entries in zip', tag, ':', len(plot_entries))
    assert plot_entries, f'no images for plot {PLOT}; folders: ' + str(
        sorted({pat.search(n).group(1) for n in z.entries if pat.search(n)})[:20])

    def doy_of(name):
        m = re.search(r'_(2014)(\d{2})(\d{2})(\d{2})(\d{2})', name)
        if not m:
            return None
        d = datetime.date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        return d.timetuple().tm_yday

    season = [n for n in plot_entries if (doy_of(n) or 0) and 113 < doy_of(n) < 150]
    sizes = sum(z.entries[n][3] for n in season) / 1e6
    print('in-window images:', len(season), f'({sizes:.0f} MB uncompressed)')

if z is not None:
    assert len(season) <= 1000, (
        f'{len(season)} season images for plot {PLOT} - unexpectedly large; stopping instead of '
        'silently downloading. Narrow the window or subsample deliberately and document it.')
    manifest_s = []
    got_s = ensure_members(tag, set(season), '/content/fcpheno/data/season_imgs', manifest_s)
    season_files = sorted(os.listdir('/content/fcpheno/data/season_imgs'))
    assert len(season_files) == len(season), (len(season_files), len(season))
else:
    fp = f'/content/fcpheno/data/full_{tag}.zip'
    if not os.path.exists(fp):
        full_download(tag, fp)
    pat = re.compile(r'_(2014)(\d{2})(\d{2})(\d{2})(\d{2})')
    os.makedirs('/content/fcpheno/data/season_imgs', exist_ok=True)
    with zipfile.ZipFile(fp) as zf:
        n_in = 0
        for m in zf.namelist():
            mm = pat.search(m)
            if not mm or f'/{PLOT}/' not in m:
                continue
            d = datetime.date(int(mm.group(1)), int(mm.group(2)), int(mm.group(3)))
            if 113 < d.timetuple().tm_yday < 150:
                zf.extract(m, '/content/fcpheno/data/_xs')
                n_in += 1
    for root, _, fs in os.walk('/content/fcpheno/data/_xs'):
        for f in fs:
            shutil.move(os.path.join(root, f), f'/content/fcpheno/data/season_imgs/{f}')
    shutil.rmtree('/content/fcpheno/data/_xs', ignore_errors=True)
    season_files = sorted(os.listdir('/content/fcpheno/data/season_imgs'))
    print('in-window images (fallback path):', len(season_files))
bad = [f for f in season_files
       if open(f'/content/fcpheno/data/season_imgs/{f}', 'rb').read(3) != b'\xff\xd8\xff']
assert not bad, f'non-JPEG payloads: {bad[:5]}'
print('season images on disk:', len(season_files))
print('first/last:', season_files[0], season_files[-1])

## 8. Input preview — labelled pixels over the raw image

**Purpose / expected result:** display one of the training images with the authors' labelled pixel positions overlaid (colour = class). This is real input data with the authors' real annotations — the classifier's ground truth. Note the visible timestamp strip in the image; the author pipeline crops the bottom 16 pixel rows before feature computation ("16 rows at the bottom of the pictures contain the timestamps", Phase_2.md).

*学習画像の1枚に、著者のラベル付けピクセルを重ねて表示します。*

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

CLASS_COLORS = {
    'Soil': 'saddlebrown', 'Green_vegetation': 'seagreen',
    'Kna_arv_flower': 'purple', 'Ran_acr_flower': 'gold',
    'Leu_vul_flower': 'cyan', 'Gra_flower': 'grey20',
}
# deterministic choice: the labelled image with the most labelled pixels
counts = labs.groupby('im').size().sort_values(ascending=False)
prev_name = counts.index[0]
img = np.asarray(Image.open(f'/content/fcpheno/data/train_imgs/{prev_name}'))
sub = labs[labs['im'] == prev_name]
fig, ax = plt.subplots(figsize=(12.8, 10.4), dpi=100)
ax.imshow(img)
for cls, g in sub.groupby('type'):
    ax.scatter(g['x'], g['y'], s=18, c=CLASS_COLORS.get(str(cls), 'red'),
               label=str(cls), edgecolors='white', linewidths=0.3)
ax.set_title(f'{prev_name} — authors labelled pixels (ground-truth input annotation)')
ax.legend(loc='lower right', fontsize=8, framealpha=0.8)
ax.axis('off')
fig.tight_layout()
fig.savefig('/content/fcpheno/out_labelled_preview.png', bbox_inches='tight')
plt.show()
print('image size:', img.shape, '| labelled pixels shown:', len(sub))

## 9. Feature extraction at labelled pixels + RF training (authors' Phase 2→3 path)

**Purpose / expected result:** run the authors' pipeline (Phase_2.md §2.1, Phase_3.md) on the labelled images at the final settings DF=4 / WS=11 and the 11 SFFS-selected bands recorded in their Phase 3/4 code: `R, G, B, rgbvi, gli, vari, ngrdi, Rsecond_moment, Bcontrast, Bentropy, Bsecond_moment`. Steps, all from the pinned author code:

1. crop to `extent(0,1280,16,1040)` (removes the timestamp strip), `terra::aggregate` by DF=4;
2. RGBVI, GLI, VARI, NGRDI (author equations; VARI infinities replaced by finite extremes, Phase_2.md §2.2);
3. `glcm()` texture with `window=c(11,11)`, `na_opt="center"`, `min_x=0,max_x=255`, 4 shifts (rotation-invariant averaging inside the `glcm` package) — only the statistics kept by the SFFS set (`second_moment` on red; `contrast, entropy, second_moment` on blue), computed on the authors' minimal cropped extent around the labelled pixels;
4. extract feature values at the labelled pixel coordinates (`raster::extract`, author calls);
5. 70 %/30 % train/validation split **by image** with `set.seed(1409)`; RF `ntree=500`, `set.seed(1509)` (author seeds);
6. per-class F1 on the validation images (manual confusion-table math replacing `crfsuite::crf_evaluation`) and the confusion table.

**Documented deviation:** `proximity=TRUE` (author Phase_3) is omitted — it only computes an N×N proximity matrix (hundreds of MB for ~7000 pixels) and has no effect on predictions.

*著者のPhase 2→3処理(最終11バンド、DF=4/WS=11、著者のシード)をRで実行し、検証画像に対するF1を報告します。*

In [ ]:
train_r = r'''suppressMessages({library(raster); library(terra); library(glcm); library(sp)})
setwd("/content/fcpheno")
lab <- read.csv(list.files("data/rois_labels", "Phase_1_labelled.csv", recursive=TRUE, full.names=TRUE)[1])
lab$im <- basename(as.character(lab$im))
df <- 4; ws <- 11
BEST <- c("R","G","B","rgbvi","gli","vari","ngrdi","Rsecond_moment","Bcontrast","Bentropy","Bsecond_moment")
feat_all <- NULL
t0 <- proc.time()[3]
for (imgname in unique(lab$im)) {
  f <- file.path("data/train_imgs", imgname)
  if (!file.exists(f)) { cat("MISSING", imgname, "\n"); flush.console(); next }
  img <- brick(f)
  img <- crop(img, extent(0, 1280, 16, 1040))
  imgr <- terra::aggregate(img, fact = df)
  labsi <- lab[lab$im == imgname, ]
  pt <- SpatialPoints(labsi[, c("x","y")])
  cellnum <- raster::extract(imgr, pt, sp = TRUE, cellnumbers = TRUE)@data[["cells"]]
  rc <- rowColFromCell(imgr, cellnum); rown <- rc[,1]; coln <- rc[,2]
  xmin <- ((min(coln) - ((ws - 1) / 2)) - 2) * df
  xmax <- ((max(coln) + ((ws - 1) / 2)) + 1) * df
  ymin <- extent(imgr)[4] - ((max(rown) + ((ws - 1) / 2)) + 1) * df
  ymax <- extent(imgr)[4] - ((min(rown) - ((ws - 1) / 2)) - 2) * df
  cropped <- crop(imgr, extent(xmin, xmax, ymin, ymax))
  cropped[[4]] <- ((cropped[[2]]*cropped[[2]]) - (cropped[[1]]*cropped[[3]])) /
                  ((cropped[[2]]*cropped[[2]]) + (cropped[[1]]*cropped[[3]]))                      # RGBVI
  cropped[[5]] <- ((2*cropped[[2]]) - cropped[[1]] - cropped[[3]]) /
                  ((2*cropped[[2]]) + cropped[[1]] + cropped[[3]])                                  # GLI
  cropped[[6]] <- (cropped[[2]] - cropped[[1]]) / (cropped[[2]] + cropped[[1]] - cropped[[3]])      # VARI
  cropped[[7]] <- (cropped[[2]] - cropped[[1]]) / (cropped[[2]] + cropped[[1]])                     # NGRDI
  glcm.red  <- glcm(cropped[[1]], window=c(ws,ws), na_opt="center", min_x=0, max_x=255,
                    shift=list(c(0,1),c(1,1),c(1,0),c(1,-1)), statistics=c("second_moment"))
  glcm.blue <- glcm(cropped[[3]], window=c(ws,ws), na_opt="center", min_x=0, max_x=255,
                    shift=list(c(0,1),c(1,1),c(1,0),c(1,-1)),
                    statistics=c("contrast","entropy","second_moment"))
  imgrs <- stack(cropped, glcm.red, glcm.blue)
  names(imgrs) <- BEST
  ptsa <- raster::extract(imgrs, pt, sp = TRUE)@data[, BEST]
  ptsa$type <- as.character(labsi$type)
  ptsa$imname <- imgname
  feat_all <- rbind(feat_all, ptsa)
  cat("features:", imgname, nrow(ptsa), "| elapsed", round(proc.time()[3]-t0,1), "s\n"); flush.console()
}
cat("total labelled feature rows:", nrow(feat_all), "\n")

# author Phase_2.2 VARI handling (on the table)
feat_all$vari <- as.numeric(feat_all$vari)
feat_all$vari[is.na(feat_all$vari)] <- 0
feat_all$vari[feat_all$vari == Inf]  <- max(feat_all$vari[is.finite(feat_all$vari)])
feat_all$vari[feat_all$vari == -Inf] <- min(feat_all$vari[is.finite(feat_all$vari)])
write.csv(feat_all, "data/Phase2_selected_features.csv", row.names = FALSE)

# author split: 70% of IMAGES (seed 1409), RF seed 1509, ntree 500
set.seed(1409)
n_img <- length(unique(feat_all$imname))
train_index <- sample(seq(1, n_img), round(n_img * 0.7))
imgst <- unique(feat_all$imname)[train_index]
feat_all$tv <- "v"; feat_all$tv[feat_all$imname %in% imgst] <- "t"
train <- feat_all[feat_all$tv == "t", ]; valid <- feat_all[feat_all$tv == "v", ]
cat("train/valid images:", length(unique(train$imname)), length(unique(valid$imname)),
    "| rows:", nrow(train), nrow(valid), "\n")
set.seed(1509)
classifier_RF <- randomForest(x = train[, BEST], y = as.factor(train$type), ntree = 500)  # proximity omitted (documented)
pred_rf <- predict(classifier_RF, newdata = valid)
cm <- table(pred = pred_rf, truth = valid$type)
print(cm)
cls <- union(rownames(cm), colnames(cm))
f1 <- sapply(cls, function(cl) {
  tp <- if (cl %in% rownames(cm) && cl %in% colnames(cm)) cm[cl, cl] else 0
  fp <- sum(cm[, cl]) - tp; fn <- sum(cm[cl, ]) - tp
  p <- tp / (tp + fp); r <- tp / (tp + fn)
  ifelse((p + r) == 0, 0, 2 * p * r / (p + r))
})
f1_mean <- mean(f1)
cat("per-class F1:\n"); print(round(f1, 4)); cat("MEAN F1 (validation images):", round(f1_mean, 4), "\n")
write.csv(data.frame(class = names(f1), f1 = as.numeric(f1), f1_mean = f1_mean),
          "data/RF_validation_f1.csv", row.names = FALSE)
saveRDS(classifier_RF, "data/RF_selected.rds")
cat("saved data/RF_selected.rds\n")
'''
open('/content/fcpheno/train_features.R', 'w').write(train_r)
p = subprocess.run(['/usr/bin/Rscript', '/content/fcpheno/train_features.R'],
                   capture_output=True, text=True)
print(p.stdout[-8000:])
if p.returncode != 0:
    print(p.stderr[-4000:]); raise RuntimeError('training R script failed')

## 10. Brightness/contrast image selection (authors' Phase 1.1–1.2, single plot)

**Purpose / expected result:** for every season image of the chosen plot, compute the authors' brightness statistics **over the ROI pixels** exactly as `phenopix::extractVIs` does — `bri.av = mean(R+G+B)`, `bri.sd = sd(R+G+B)` (verified in the phenopix source, `extractVIs.R` lines 89–90; no aggregation, `npixels=1` as in the authors' call). Then apply the authors' selection rule (Phase_1.md §1.2): within each 3-day window (`doy-2 < x < doy+2`), keep images whose brightness **and** contrast percentile lie strictly between 0.1 and 0.4; images matching both criteria (`selBRICON`) are the FCTS inputs.

**ROI orientation:** `extractVIs` masks a `flip()`ped brick, while the authors' Phase_4 classification masks a non-flipped brick with the same polygon. This cell determines the ROI polygon's actual convention empirically: the camera's timestamp strip occupies the bottom 16 image rows, which a vegetation ROI cannot include — the orientation whose masked region stays clear of rows 1025–1040 is used (decision and evidence printed). If both are consistent, the authors' literal Phase_4 convention (non-flipped) is used everywhere.

*選択プロットの全季節画像について、ROI内の明るさ統計量を算出し、著者の3日窓・パーセンタイル規則でFCTS入力画像を選択します。ROI多角形の座標系はタイムスタンプ帯を含まない向きで実測により決定します。*

In [ ]:
select_r = r'''suppressMessages({library(raster); library(terra); library(sp)})
setwd("/content/fcpheno")
args <- commandArgs(TRUE)
plot_id <- args[1]                      # e.g. "010"
roi_file <- args[2]
out_csv <- args[3]
load(roi_file)
pol <- roi.data[[1]]$polygons

imgdir <- "data/season_imgs"
files <- sort(list.files(imgdir, pattern="\\.jpg$"))
get_date <- function(f) {
  m <- regmatches(f, regexec("_(2014)([0-9]{2})([0-9]{2})([0-9]{2})([0-9]{2})", f))[[1]]
  as.POSIXct(sprintf("%s-%s-%s %s:%s", m[2], m[3], m[4], m[5], m[6]), format="%Y-%m-%d %H:%M", tz="UTC")
}
dates <- sapply(files, get_date)
doys <- as.numeric(format(dates, "%j"))
keep <- which(!is.na(doys) & doys > 113 & doys < 150)
files <- files[keep]; dates <- dates[keep]; doys <- doys[keep]
cat("season images to score:", length(files), "\n")

# decide ROI orientation once: masked region must exclude image rows 1025:1040 (timestamp strip)
probe_f <- files[1]
B <- brick(file.path(imgdir, probe_f)); Fk <- raster::flip(B)
rb <- rowColFromCell(B, Which(!is.na(mask(B[[1]], pol)), cells = TRUE))[, 1]
rf_ <- rowColFromCell(Fk, Which(!is.na(mask(Fk[[1]], pol)), cells = TRUE))[, 1]
stripB <- any(rb >= 1025); stripF <- any(rf_ >= 1025)
FLIP <- if (!stripF && stripB) TRUE else if (!stripB && stripF) FALSE else FALSE
cat(sprintf("orientation: B-mask hits strip=%s, F-mask hits strip=%s -> use flip=%s\n",
            stripB, stripF, FLIP))
if (stripB == stripF) cat("NOTE: ambiguous orientation; using the authors' literal Phase_4 convention (no flip).\n")
writeLines(as.character(FLIP), "data/flip_flag.txt")

rows <- NULL
t0 <- proc.time()[3]
for (i in seq_along(files)) {
  r <- brick(file.path(imgdir, files[i]))
  if (FLIP) r <- raster::flip(r)
  m <- mask(r[[1]], pol)
  sel <- !is.na(values(m))
  b <- values(r[[1]])[sel] + values(r[[2]])[sel] + values(r[[3]])[sel]
  rows <- rbind(rows, data.frame(imname = files[i], date = format(dates[i], "%Y-%m-%d %H:%M"),
                                 doy = doys[i], roi_px = sum(sel),
                                 bri.av = mean(b), bri.sd = sd(b)))
  if (i %% 25 == 0) { cat("scored", i, "/", length(files), "| elapsed",
                          round(proc.time()[3]-t0,1), "s\n"); flush.console() }
}
dataraw <- rows
# authors' 3-day-window percentile rule (Phase_1.md 1.2)
perc <- c(0.1, 0.4)
selBRI <- rep(FALSE, nrow(dataraw)); selCON <- rep(FALSE, nrow(dataraw))
for (d in unique(dataraw$doy)) {
  idx <- which(dataraw$doy > d - 2 & dataraw$doy < d + 2)
  pb <- ecdf(dataraw$bri.av[idx])(dataraw$bri.av[idx])
  pc <- ecdf(dataraw$bri.sd[idx])(dataraw$bri.sd[idx])
  selBRI[idx] <- pb > perc[1] & pb < perc[2]
  selCON[idx] <- pc > perc[1] & pc < perc[2]
}
dataraw$selBRI <- selBRI; dataraw$selCON <- selCON
dataraw$selBRICON <- dataraw$selBRI & dataraw$selCON
# de-duplicate by date (author step)
dataraw$selBRICON[duplicated(dataraw$date[dataraw$selBRICON])] <- FALSE
cat("selected images (selBRICON):", sum(dataraw$selBRICON), "of", nrow(dataraw), "\n")
write.csv(dataraw, out_csv, row.names = FALSE)
'''
import glob, json
roi_files = glob.glob(f'/content/fcpheno/data/rois_labels/**/{PLOT}/roi.data.Rdata', recursive=True)
assert roi_files, f'roi.data.Rdata for plot {PLOT} not found'
open('/content/fcpheno/select_images.R', 'w').write(select_r)
p = subprocess.run(['/usr/bin/Rscript', '/content/fcpheno/select_images.R', PLOT, roi_files[0],
                    '/content/fcpheno/data/selection_010.csv' if PLOT == '010' else '/content/fcpheno/data/selection.csv'],
                   capture_output=True, text=True)
print(p.stdout[-6000:])
if p.returncode != 0:
    print(p.stderr[-4000:]); raise RuntimeError('selection R script failed')
import pandas as pd
sel_csv = '/content/fcpheno/data/selection.csv' if PLOT != '010' else '/content/fcpheno/data/selection_010.csv'
sel = pd.read_csv(sel_csv)
print(sel[['selBRI','selCON','selBRICON']].sum())
sel[sel.selBRICON][['imname','date','bri.av','bri.sd']].head(10)

## 11. Classify selected images and compute per-image flower cover (authors' Phase 4.1)

**Purpose / expected result:** for every `selBRICON` image: aggregate by DF=4, compute the 11 bands, **mask with the plot's ROI polygon** (`mask(stack(...), roi.data[[1]]$polygons)`, author Phase_4.md), `terra::predict` with the trained RF, and compute each class's percentage of non-NA ROI pixels (author §4.1). Each classified raster is stored (compressed, ~17 MB total) for the visualization in §13. Expected runtime: seconds per image; progress is streamed.

Note on the authors' tutorial: Phase_4.md re-reads the *global* selected-image list inside the per-plot loop; `image_classification.md` filters it by plot (`substr(imname,8,10)==plot`). We follow the filtered, per-plot interpretation and note it.

*選択画像を各クラスに分類し、ROI内の非NAピクセルに占める各クラスの割合(%)を計算します(著者Phase 4.1の移植)。*

In [ ]:
classify_r = r'''suppressMessages({library(raster); library(terra); library(glcm); library(sp)})
setwd("/content/fcpheno")
args <- commandArgs(TRUE)
plot_id <- args[1]; roi_file <- args[2]; sel_csv <- args[3]; out_csv <- args[4]
rf <- readRDS("data/RF_selected.rds")
BEST <- c("R","G","B","rgbvi","gli","vari","ngrdi","Rsecond_moment","Bcontrast","Bentropy","Bsecond_moment")
CLASSES <- c("Soil","Green_vegetation","Kna_arv_flower","Ran_acr_flower","Leu_vul_flower","Gra_flower")
df <- 4; ws <- 11
FLIP <- readLines("data/flip_flag.txt")[1] == "TRUE"
load(roi_file); pol <- roi.data[[1]]$polygons
sel <- read.csv(sel_csv)
seli <- sel[sel$selBRICON, ]
cat("images to classify:", nrow(seli), "| flip:", FLIP, "\n")
dir.create("classified", showWarnings = FALSE)

cover <- NULL
t0 <- proc.time()[3]
for (i in seq_len(nrow(seli))) {
  img <- brick(file.path("data/season_imgs", seli$imname[i]))
  if (FLIP) img <- raster::flip(img)
  imgr <- terra::aggregate(img, fact = df)
  imgr[[4]] <- ((imgr[[2]]*imgr[[2]]) - (imgr[[1]]*imgr[[3]])) /
               ((imgr[[2]]*imgr[[2]]) + (imgr[[1]]*imgr[[3]]))                                # RGBVI
  imgr[[5]] <- ((2*imgr[[2]]) - imgr[[1]] - imgr[[3]]) /
               ((2*imgr[[2]]) + imgr[[1]] + imgr[[3]])                                        # GLI
  imgr[[6]] <- (imgr[[2]] - imgr[[1]]) / (imgr[[2]] + imgr[[1]] - imgr[[3]])                  # VARI
  imgr[[6]][imgr[[6]] == Inf]  <- max(getValues(imgr[[6]])[is.finite(getValues(imgr[[6]]))])  # author Inf handling
  imgr[[6]][imgr[[6]] == -Inf] <- min(getValues(imgr[[6]])[is.finite(getValues(imgr[[6]]))])
  imgr[[7]] <- (imgr[[2]] - imgr[[1]]) / (imgr[[2]] + imgr[[1]])                              # NGRDI
  glcm.red  <- glcm(imgr[[1]], window=c(ws,ws), na_opt="center", min_x=0, max_x=255,
                    shift=list(c(0,1),c(1,1),c(1,0),c(1,-1)), statistics=c("second_moment"))
  glcm.blue <- glcm(imgr[[3]], window=c(ws,ws), na_opt="center", min_x=0, max_x=255,
                    shift=list(c(0,1),c(1,1),c(1,0),c(1,-1)),
                    statistics=c("contrast","entropy","second_moment"))
  imgm <- mask(stack(imgr, glcm.red, glcm.blue), pol)
  names(imgm) <- BEST
  classified <- terra::predict(imgm, rf, na.rm = TRUE)
  saveRDS(as.integer(as.vector(values(classified))),
          file.path("classified", sub("\\.jpg$", "", seli$imname[i])), compress = "gzip")
  fr <- freq(classified)
  fr <- fr[!is.na(fr[, "value"]), , drop = FALSE]
  notNA <- sum(fr[, "count"])
  row <- data.frame(imname = seli$imname[i], date = seli$date[i])
  for (k in seq_along(CLASSES)) {
    v <- fr[fr[, "value"] == k, "count"]
    row[[CLASSES[k]]] <- if (length(v)) v / notNA else NA
  }
  cover <- rbind(cover, row)
  if (i %% 10 == 0 || i == nrow(seli)) {
    cat("classified", i, "/", nrow(seli), "| elapsed", round(proc.time()[3]-t0, 1), "s\n")
    flush.console()
  }
}
cover$plot <- plot_id
write.csv(cover, out_csv, row.names = FALSE)
cat("wrote", out_csv, nrow(cover), "rows\n")
'''
import glob
roi_files = glob.glob(f'/content/fcpheno/data/rois_labels/**/{PLOT}/roi.data.Rdata', recursive=True)
open('/content/fcpheno/classify_plot.R', 'w').write(classify_r)
p = subprocess.run(['/usr/bin/Rscript', '/content/fcpheno/classify_plot.R', PLOT, roi_files[0],
                    '/content/fcpheno/data/selection.csv' if PLOT != '010' else '/content/fcpheno/data/selection_010.csv',
                    '/content/fcpheno/data/dataflowers.csv'],
                   capture_output=True, text=True)
print(p.stdout[-8000:])
if p.returncode != 0:
    print(p.stderr[-4000:]); raise RuntimeError('classification R script failed')

## 12. Classified image vs. original RGB (peak-flowering date)

**Purpose / expected result:** render the original camera image next to the RF classification for the date with the maximum sown-flower cover from the table just computed. Class colours are the authors' palette from `image_classification.md` (class order 1–6 = `Gra_flower, Green_vegetation, Kna_arv_flower, Leu_vul_flower, Ran_acr_flower, Soil` → grey20, seagreen, purple, cyan, gold, brown). The classification is recomputed for that single date with the same code path as §11 (author Phase_4: ROI-masked stack → `terra::predict`), and the classified matrix is written to CSV; the panel rendering happens here in Python so image orientation is explicit. This is the paper's Figure-5-style comparison — model prediction, clearly labelled as such.

*ピーク日の分類結果(モデル予測)と原本RGBを並べて表示します(著者のパレット・クラス順)。*

In [ ]:
viz_r = r'''suppressMessages({library(raster); library(terra); library(sp)})
setwd("/content/fcpheno")
args <- commandArgs(TRUE)
imgname <- args[1]; out_csv <- args[2]; roi_file <- args[3]
rf <- readRDS("data/RF_selected.rds")
BEST <- c("R","G","B","rgbvi","gli","vari","ngrdi","Rsecond_moment","Bcontrast","Bentropy","Bsecond_moment")
df <- 4; ws <- 11
FLIP <- readLines("data/flip_flag.txt")[1] == "TRUE"
load(roi_file); pol <- roi.data[[1]]$polygons
img <- brick(file.path("data/season_imgs", imgname))
if (FLIP) img <- raster::flip(img)
imgr <- terra::aggregate(img, fact = df)
imgr[[4]] <- ((imgr[[2]]*imgr[[2]]) - (imgr[[1]]*imgr[[3]])) / ((imgr[[2]]*imgr[[2]]) + (imgr[[1]]*imgr[[3]]))
imgr[[5]] <- ((2*imgr[[2]]) - imgr[[1]] - imgr[[3]]) / ((2*imgr[[2]]) + imgr[[1]] + imgr[[3]])
imgr[[6]] <- (imgr[[2]] - imgr[[1]]) / (imgr[[2]] + imgr[[1]] - imgr[[3]])
imgr[[6]][imgr[[6]] == Inf]  <- max(getValues(imgr[[6]])[is.finite(getValues(imgr[[6]]))])
imgr[[6]][imgr[[6]] == -Inf] <- min(getValues(imgr[[6]])[is.finite(getValues(imgr[[6]]))])
imgr[[7]] <- (imgr[[2]] - imgr[[1]]) / (imgr[[2]] + imgr[[1]])
glcm.red  <- glcm(imgr[[1]], window=c(ws,ws), na_opt="center", min_x=0, max_x=255,
                  shift=list(c(0,1),c(1,1),c(1,0),c(1,-1)), statistics=c("second_moment"))
glcm.blue <- glcm(imgr[[3]], window=c(ws,ws), na_opt="center", min_x=0, max_x=255,
                  shift=list(c(0,1),c(1,1),c(1,0),c(1,-1)),
                  statistics=c("contrast","entropy","second_moment"))
imgm <- mask(stack(imgr, glcm.red, glcm.blue), pol)
names(imgm) <- BEST
classified <- terra::predict(imgm, rf, na.rm = TRUE)
M <- matrix(as.integer(as.vector(values(classified))), nrow(classified), ncol(classified))
if (FLIP) M <- M[nrow(M):1, ]     # raster rows are bottom-up when flipped; store top-down
write.table(M, out_csv, sep = ",", row.names = FALSE, col.names = FALSE)
cat("wrote", out_csv, dim(M)[1], "x", dim(M)[2], "\n")
'''
import glob
roi_files = glob.glob(f'/content/fcpheno/data/rois_labels/**/{PLOT}/roi.data.Rdata', recursive=True)
cover = pd.read_csv('/content/fcpheno/data/dataflowers.csv')
sown_candidates = [c for c in ['Leu_vul_flower','Kna_arv_flower','Ran_acr_flower','Gra_flower'] if c in cover.columns]
sprow = str(comp[comp['_pid'] == PLOT][spcol].astype(str).iloc[0])
sown_flag = {'Leu_vul_flower': 'Leucanthemum' in sprow, 'Kna_arv_flower': 'Knautia' in sprow,
             'Ran_acr_flower': 'Ranunculus' in sprow,
             'Gra_flower': any(s in sprow for s in ['Festuca','Avenula','Poa','Anthoxanthum','Phleum','Dactylis','Holcus'])}
sown_here = [c for c in sown_candidates if sown_flag.get(c, False)]
print('sown flower classes for plot', PLOT, ':', sown_here)
assert sown_here, 'chosen plot has no sown flower class; pick another plot deliberately'
peak_col = max(sown_here, key=lambda c: cover[c].max())
peak_row = cover.loc[cover[peak_col].idxmax()]
print('peak sown class:', peak_col, '| peak cover %.3f on' % peak_row[peak_col], peak_row['date'])
open('/content/fcpheno/viz_classified.R', 'w').write(viz_r)
p = subprocess.run(['/usr/bin/Rscript', '/content/fcpheno/viz_classified.R', peak_row['imname'],
                    '/content/fcpheno/data/classified_matrix.csv', roi_files[0]],
                   capture_output=True, text=True)
print(p.stdout[-2000:])
if p.returncode != 0:
    print(p.stderr[-3000:]); raise RuntimeError('classified-matrix R script failed')

import numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from PIL import Image
img = np.asarray(Image.open(f"/content/fcpheno/data/season_imgs/{peak_row['imname']}"))
M = pd.read_csv('/content/fcpheno/data/classified_matrix.csv', header=None).values
CLASSES = ['Gra_flower','Green_vegetation','Kna_arv_flower','Leu_vul_flower','Ran_acr_flower','Soil']
cols = ['grey20','seagreen','purple','cyan','gold','brown']   # author palette, class order 1..6
fig, axes = plt.subplots(1, 2, figsize=(19.2, 7.8), dpi=100)
axes[0].imshow(img)
axes[0].set_title(f"Original RGB — {peak_row['imname']}")
axes[0].axis('off')
axes[1].imshow(np.ma.masked_invalid(M), cmap=ListedColormap(cols), vmin=0.5, vmax=6.5,
               interpolation='nearest')
axes[1].set_title('RF classification (model prediction, ROI-masked)')
axes[1].axis('off')
handles = [plt.Rectangle((0, 0), 1, 1, fc=c) for c in cols]
axes[1].legend(handles, CLASSES, loc='lower left', fontsize=8, framealpha=0.85)
fig.tight_layout()
fig.savefig('/content/fcpheno/out_classified.png', bbox_inches='tight')
plt.show()
print('classified matrix:', M.shape, '| non-NA cells:', int((~np.isnan(M)).sum()))

## 13. FCTS smoothing (authors' Phase 4.2)

**Purpose / expected result:** apply the authors' smoothing chain to the per-image cover table: `forecast::tsclean` outlier removal per class (Friedman's supersmoother based, `replace.missing=FALSE`, author call), daily means, and a `stats::loess` fit with standard errors per class; 95 % pointwise intervals from `fit ± qt(0.975, df)·se.fit` (author §4.2). The plot below is generated here from this run's outputs (it is a notebook visualization of the reproduced quantities, not a copy of a paper figure).

*各クラスのFCTSにtsclean→日次平均→loess平滑を適用し、信頼区間付きでプロットします。*

In [ ]:
smooth_r = r'''suppressMessages({library(forecast); library(zoo); library(dplyr)})
setwd("/content/fcpheno")
classes <- c("Gra_flower","Kna_arv_flower","Leu_vul_flower","Ran_acr_flower","Green_vegetation","Soil")
ts <- read.csv("data/dataflowers.csv", stringsAsFactors = FALSE)
ts[is.na(ts)] <- 0   # author: NA assigned to classes with zero pixels
# equivalent to the author's read.zoo(..., tryFormats=...) for our "%Y-%m-%d %H:%M" dates
idx <- as.POSIXct(ts$date, format = "%Y-%m-%d %H:%M")
tszo <- zoo(ts[, classes], order.by = idx)
tot_df <- NULL
for (cl in classes) {
  z <- tsclean(tszo[, cl], replace.missing = FALSE)   # author call
  temp_df <- data.frame(Index = time(z), FPe = as.numeric(z))
  temp_df$class <- cl
  tot_df <- rbind(tot_df, temp_df)
}
tot_df$date <- as.Date(tot_df$Index)
tot_daily <- tot_df %>% group_by(date, class) %>% summarise(FPe = mean(FPe), .groups = "drop")
fitted_df <- NULL
for (cl in classes) {
  temp_df <- tot_daily[tot_daily$class == cl, ]
  if (nrow(temp_df) >= 4) {   # loess needs a minimum number of daily points
    pr <- predict(loess(FPe ~ as.numeric(date), data = temp_df), se = TRUE)
    fitted_df <- rbind(fitted_df, cbind(temp_df, fit = pr$fit, se.fit = pr$se.fit, df = pr$df))
  }
}
fitted_df$lower <- fitted_df$fit - qt(0.975, fitted_df$df) * fitted_df$se.fit
fitted_df$upper <- fitted_df$fit + qt(0.975, fitted_df$df) * fitted_df$se.fit
fitted_df$plot <- read.csv("data/dataflowers.csv")$plot[1]
write.csv(fitted_df, "data/fitted_flower_cover.csv", row.names = FALSE)
cat("fitted rows:", nrow(fitted_df), "\n")

png("out_fcts.png", width = 2200, height = 1250, res = 150)
par(mar = c(4.5, 4.5, 3, 8.5))
plot(range(fitted_df$date), range(0, fitted_df$upper), type = "n",
     xlab = "Date (2014)", ylab = "Flower/cover fraction of ROI pixels",
     main = sprintf("Flower cover time series - plot %s (this run, smoothed)", fitted_df$plot[1]))
pal <- c(Gra_flower = "grey20", Kna_arv_flower = "purple", Leu_vul_flower = "cyan",
         Ran_acr_flower = "gold", Green_vegetation = "seagreen", Soil = "brown")
for (cl in classes) {
  d <- fitted_df[fitted_df$class == cl, ]
  if (!nrow(d)) next
  polygon(c(d$date, rev(d$date)), c(d$lower, rev(d$upper)),
          col = adjustcolor(unname(pal[cl]), 0.15), border = NA)
  lines(d$date, d$fit, col = unname(pal[cl]), lwd = 2)
}
abline(h = 0.01, lty = 3, col = "grey50")
legend("right", legend = names(pal), fill = pal, inset = c(-0.32, 0), xpd = NA, cex = 0.8)
dev.off()
cat("saved out_fcts.png\n")
'''
open('/content/fcpheno/smooth_fcts.R', 'w').write(smooth_r)
p = subprocess.run(['/usr/bin/Rscript', '/content/fcpheno/smooth_fcts.R'], capture_output=True, text=True)
print(p.stdout[-4000:])
if p.returncode != 0:
    print(p.stderr[-4000:]); raise RuntimeError('smoothing R script failed')
from IPython.display import Image as IPyImage
IPyImage(filename='/content/fcpheno/out_fcts.png')

## 14. Phenological metrics: onset / peak / end (authors' Phase 4.3)

**Purpose / expected result:** apply the authors' `findpm` rule to the smoothed series of the plot's **sown** flower classes (composition from §4): peak = day of maximum; onset = first day with normalized cumulative sum (before peak) > 0.1; end = first day with normalized cumulative sum (after peak) > 0.9; metrics suppressed when the sown peak cover < 1 %, when start/end cover > 1 %, or when the peak is at the series edge (author guards). Results may legitimately be `NA` when a guard applies — that is the authors' intended behaviour, not a failure.

*平滑化FCTSから著者の規則で開花開始・ピーク・終了日を抽出します(ガード条件は著者仕様)。*

In [ ]:
metrics_r = r'''suppressMessages({library(dplyr); library(scales)})
setwd("/content/fcpheno")
classes_flower <- c("Gra_flower","Kna_arv_flower","Leu_vul_flower","Ran_acr_flower")
fall <- read.csv("data/fitted_flower_cover.csv", stringsAsFactors = TRUE)
fall$date <- as.Date(fall$date)
fall <- fall[fall$class %in% classes_flower, ]
compo <- read.csv(list.files("data/rois_labels", pattern="omposition.*\\.csv$", recursive=TRUE, full.names=TRUE)[1])
pcol <- if ("plot" %in% names(compo)) "plot" else grep("plot", names(compo), value=TRUE)[1]
compo$Ran_acr_flower <- grepl("Ranunculus", compo$sp_list)
compo$Leu_vul_flower <- grepl("Leucanthemum", compo$sp_list)
compo$Kna_arv_flower <- grepl("Knautia", compo$sp_list)
compo$Gra_flower <- grepl("Festuca|Avenula|Poa|Anthoxanthum|Phleum|Dactylis|Holcus", compo$sp_list)
compo$plot_key <- sprintf("%03d", compo[[pcol]])
fall$plot_key <- sprintf("%03d", as.numeric(fall$plot))
merged_df <- merge(fall, compo, by.x = "plot_key", by.y = "plot_key")
class_index <- match(merged_df$class, classes_flower)
merged_df$sown <- merged_df[, classes_flower][cbind(1:nrow(merged_df), class_index)]

limitonset <- 0.1; limitend <- 0.9
findpm <- function(df) {
  onset <- peak <- end <- NA
  dates <- df$date; vec <- df$fit
  if (length(vec) < 3) return(c(NA, NA, NA))
  pk <- which.max(vec)
  dfbeforepeak <- data.frame(date = dates[1:pk], vec = vec[1:pk])
  dfbeforepeak$rescaled <- rescale(cumsum(dfbeforepeak$vec))
  dfafterpeak <- data.frame(date = dates[pk:length(vec)], vec = vec[pk:length(vec)])
  dfafterpeak$rescaled <- rescale(cumsum(dfafterpeak$vec))
  onset <- dfbeforepeak$date[which(dfbeforepeak$rescaled > limitonset)[1]]
  peak <- dates[pk]
  end <- dfafterpeak$date[which(dfafterpeak$rescaled > limitend)[1]]
  pm <- c(onset, peak, end)
  if (max(vec) < 0.01) pm <- c(NA, NA, NA)
  if (vec[1] > 0.01) pm[1] <- NA
  if (vec[length(vec)] > 0.01) pm[3] <- NA
  if (peak == max(dates) || peak == min(dates)) pm <- c(NA, NA, NA)
  return(pm)
}
phenometrics <- merged_df %>% group_by(plot_key, class, sown) %>%
  summarise(onset = findpm(pick(everything()))[1],
            peak = findpm(pick(everything()))[2],
            end = findpm(pick(everything()))[3], .groups = "drop")
print(as.data.frame(phenometrics))
write.csv(as.data.frame(phenometrics), "data/phenometrics.csv", row.names = FALSE)
cat("saved data/phenometrics.csv\n")
'''
open('/content/fcpheno/phenometrics.R', 'w').write(metrics_r)
p = subprocess.run(['/usr/bin/Rscript', '/content/fcpheno/phenometrics.R'], capture_output=True, text=True)
print(p.stdout[-5000:])
if p.returncode != 0:
    print(p.stderr[-4000:]); raise RuntimeError('phenometrics R script failed')

## 15. Results table and CSV export

**Purpose / expected result:** a small per-date table of the reproduced flower-cover values (all classes, % of ROI pixels), plus the saved CSVs. In Colab, right-click any file under `/content/fcpheno/data/` to download it.

*再現された被覆率の表とCSVを出力します。*

In [ ]:
import pandas as pd
pd.set_option('display.width', 160)
cover = pd.read_csv('/content/fcpheno/data/dataflowers.csv')
cols = ['date'] + ['Soil','Green_vegetation','Kna_arv_flower','Ran_acr_flower','Leu_vul_flower','Gra_flower']
show = cover[cols].copy()
for c in cols[1:]:
    show[c] = (show[c] * 100).round(2)
print(f'Per-image class cover (% of ROI pixels), plot {PLOT}, {len(show)} selected images:')
print(show.iloc[::max(1, len(show)//15)].to_string(index=False))
show.to_csv('/content/fcpheno/data/flower_cover_percent.csv', index=False)
print('\nCSV files written under /content/fcpheno/data/:')
import os
for f in sorted(os.listdir('/content/fcpheno/data')):
    if f.endswith('.csv'):
        print('  ', f, os.path.getsize(f'/content/fcpheno/data/{f}'), 'bytes')

## 16. Interpretation, validation record, references

**What this run demonstrates.** The authors' four-phase pipeline — labelled-pixel features → random-forest pixel classification → ROI-masked per-image flower cover → smoothed FCTS and phenometrics — executed end-to-end for one plot and one season, using their pinned R code, their parameter values (DF=4, WS=11, 11 SFFS-selected bands, `ntree=500`, seeds 1409/1509) and their shared labelled dataset. The validation F1 above is computed on the authors' 30 % held-out images.

**Scope and limitations.**
- One plot, one season (DOY 113–150 of 2014): **not** a reproduction of the paper's 89-plot results or its reported accuracies.
- No DF–WS sweep and no SFFS re-selection: the authors' recorded settings were used (cited in their Phase 3/4 code).
- Brightness/contrast image selection was reimplemented on ROI pixels (formula verified against `phenopix::extractVIs` source, lines 89–90); `crfsuite::crf_evaluation` was replaced by manual confusion-table F1; `proximity=TRUE` omitted (no effect on predictions). All deviations are stated above.
- The code repository carries no LICENSE file; the dataset is CC BY-ND 4.0. This notebook runs the authors' pinned code without redistributing it.

**Validation record.** Filled in below from the actual run outputs (hardware, duration, package versions, and key numbers are printed by the cells above and repeated here after execution).

**References.**
1. Andreatta et al. 2023, *RSE* 298:113835, DOI 10.1016/j.rse.2023.113835.
2. Author tutorial repository `andreattad/Flower_covers_phenocams` @ `7cf58731236438ad1200aac2553c9e48461ee9bf` (Phase_1–4.md, image_classification.md; accessed 2026-10-09).
3. Research data, ETH Research Collection, DOI 10.3929/ethz-b-000634004 (CC BY-ND 4.0).
4. Filippa et al. 2016, *phenopix*; Zvoleff 2020, *glcm*; Liaw & Wiener 2002, *randomForest* (packages used by the authors).

*本runの範囲・限界・検証記録は上記の通りです。単一プロット・単一季節の部分的な再現であり、論文の89プロット結果や報告精度の再現ではありません。*